In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import json
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from src.Skeleton_model.stgcn import STGCN
from src.XAI.rise import RISE
from src.rwf2000 import RWF2000Dataset, RWF2000PoseDataset
from src.config import POSE_DATASET_ROOT_OCSORT2
from src.config import CHECKPOINT_DIR
from src.XAI.plotting_functions import plot_gradcam_over_rgb, plot_single_frame_gradcam, animate_saliency_over_rgb
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from scripts.common.get_device import get_available_device


dataset_root = POSE_DATASET_ROOT_OCSORT2

experiment_root = CHECKPOINT_DIR / "STGCN_V1" / "final_test_amsgrad" / "amsgrad_false"
device = get_available_device()

with open(experiment_root / "config.json", "r") as f:
    hyperparameters = json.load(f)

radii_dataset = RWF2000PoseDataset(dataset_root, split="train", max_people=hyperparameters["max_people"], score_mode=hyperparameters["score_mode"])
radii = compute_joint_distance_to_center_of_gravity(radii_dataset)
skeleton_graph = SkeletonGraph(radii, normalisation=hyperparameters["adjacency_normalisation_mode"])
model = STGCN(skeleton_graph.A, temporal_kernel_size=hyperparameters["temporal_kernel_size"], dropout=hyperparameters["dropout"], edge_importance_weighting=hyperparameters["edge_importance_weighting"], people_aggregation=hyperparameters["people_aggregation"]).to(device)

checkpoint = torch.load(experiment_root / "best_model.pt", map_location=device)

if "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.to(device)
model.eval()

val_dataset = RWF2000PoseDataset(dataset_root, split="val", max_people=hyperparameters["max_people"], score_mode=hyperparameters["score_mode"])

video_num = 300
video, label = val_dataset[video_num]
video_batch = video.unsqueeze(0).to(device)
# [1, 3, 150, 17, 4]

label = label.item()

Using cuda:3 with 20.26 GB free


/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


In [12]:
from src.XAI.rise import SkeletonRise
video_num = 0
video, label = val_dataset[video_num]
video_batch = video.unsqueeze(0).to(device)
# [1, 3, 150, 17, 4]

label = label.item()
rise = SkeletonRise(model)

In [13]:
output = rise.generate_heatmap(video_batch)

processed 8/1000 masks
processed 16/1000 masks
processed 24/1000 masks
processed 32/1000 masks
processed 40/1000 masks
processed 48/1000 masks
processed 56/1000 masks
processed 64/1000 masks
processed 72/1000 masks
processed 80/1000 masks
processed 88/1000 masks
processed 96/1000 masks
processed 104/1000 masks
processed 112/1000 masks
processed 120/1000 masks
processed 128/1000 masks
processed 136/1000 masks
processed 144/1000 masks
processed 152/1000 masks
processed 160/1000 masks
processed 168/1000 masks
processed 176/1000 masks
processed 184/1000 masks
processed 192/1000 masks
processed 200/1000 masks
processed 208/1000 masks
processed 216/1000 masks
processed 224/1000 masks
processed 232/1000 masks
processed 240/1000 masks
processed 248/1000 masks
processed 256/1000 masks
processed 264/1000 masks
processed 272/1000 masks
processed 280/1000 masks
processed 288/1000 masks
processed 296/1000 masks
processed 304/1000 masks
processed 312/1000 masks
processed 320/1000 masks
processed 328

In [14]:
normalised_saliency_map = output[1]

In [15]:
normalised_saliency_map.shape

torch.Size([150, 17, 4])

In [16]:
normalised_saliency_map[:, :, 3]

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:3')